# Hausman test

In [13]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

In [14]:
# CSVファイルの読み込み
data = pd.read_csv("../data/8-sem.csv")

In [15]:
# データを確認（先頭5件）
data.head()

,Year,GDP,I,GOV,M,TB
0,1970,1035.6,150.2,115.9,628.1,6.562
1,1971,1125.4,176.0,117.1,712.7,4.511
2,1972,1237.3,205.6,125.1,805.2,4.466
3,1973,1382.6,242.9,128.2,861.0,7.178
4,1974,1496.9,245.6,139.9,908.5,7.926


In [16]:
# ラグ変数の作成
data = data.sort_values('Year')
data['L_GDP'] = data['GDP'].shift(1)
data['L_M']   = data['M'].shift(1)
data['L_I']   = data['I'].shift(1)
data['L_GOV'] = data['GOV'].shift(1)
data['L_TB']  = data['TB'].shift(1)

In [19]:
# 欠損行を削除
data = data.dropna(subset=['L_GDP', 'L_M', 'L_I', 'L_GOV', 'L_TB', 'GDP', 'M', 'I', 'GOV', 'TB'])

In [20]:
# OLS
X = sm.add_constant(data[['I','GOV','TB','L_GDP','L_M','L_I','L_GOV','L_TB']])
y = data['M']
model_stage1 = sm.OLS(y, X).fit()

In [21]:
# 結果の出力
print(model_stage1.summary())

                            OLS Regression Results                            
Dep. Variable:                      M   R-squared:                       0.999
Model:                            OLS   Adj. R-squared:                  0.999
Method:                 Least Squares   F-statistic:                     2842.
Date:                   日, 09 11 2025   Prob (F-statistic):           1.96e-22
Time:                        15:53:38   Log-Likelihood:                -111.99
No. Observations:                  24   AIC:                             242.0
Df Residuals:                      15   BIC:                             252.6
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         22.4429     42.099      0.533      0.6

In [23]:
# 残差を保存
data['residM'] = model_stage1.resid

In [25]:
# GDPをMとresidMで説明
X2 = sm.add_constant(data[['M','I','GOV','residM']])
y2 = data['GDP']

model_stage2 = sm.OLS(y2, X2).fit()

In [27]:
# 結果の出力
print(model_stage2.summary())

                            OLS Regression Results                            
Dep. Variable:                    GDP   R-squared:                       0.993
Model:                            OLS   Adj. R-squared:                  0.991
Method:                 Least Squares   F-statistic:                     631.9
Date:                   日, 09 11 2025   Prob (F-statistic):           6.46e-20
Time:                        16:23:44   Log-Likelihood:                -155.35
No. Observations:                  24   AIC:                             320.7
Df Residuals:                      19   BIC:                             326.6
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       -209.0064     92.777     -2.253      0.0